> 请点击获取[课程 PPT 内容](https://www.canva.cn/design/DAG9KgDFL-g/JuY78K8O2tquhxpDmVhlSA/view?utm_content=DAG9KgDFL-g&utm_campaign=designshare&utm_medium=link2&utm_source=uniquelinks&utlId=h89f16422ac)。


# 1. 环境配置

## 1.1 python 环境准备

In [1]:
! pip install "langgraph-cli[inmem]" openai==2.11.0 dashscope==1.25.4 langchain-classic==1.0.0 langchain==1.2.0 langchain-community==0.4.1 langchain-openai==1.1.6 arxiv==2.3.1

Looking in indexes: https://pypi.tuna.tsinghua.edu.cn/simple


## 1.2 大模型密钥准备

请根据第一章内容获取相关平台的 API KEY，如若未在系统变量中填入，请将 API_KEY 信息写入以下代码（若已设置请忽略）：

In [2]:
import os

# os.environ["OPENAI_API_KEY"] = "sk-xxxxxxxx"
# os.environ["DASHSCOPE_API_KEY"] = "sk-yyyyyyyy"

# 2. Skills 模式

## 2.1 简介

在下面的任务中，将要完成的是一个 SQL 助手智能体，用来帮助在大型企业中编写跨不同业务垂直领域的 SQL 查询。在这样的企业环境里，可能存在以下两种情况之一：
- 每个业务垂直领域（vertical）都有各自独立的数据存储系统；
- 或者整个企业使用的是一个单体数据库，其中包含成千上万张表。

相比于以往直接把所有数据库 schema 塞进系统提示词中，Skills 用的是 Progressive Disclosure（渐进披露/按需加载上下文） 的思路：
- 智能体一开始只知道“有哪些技能（skills）”，每个技能只有一句话描述（很轻量）。
- 当用户的问题涉及某个业务域（比如销售分析、库存管理），智能体会通过 工具调用（tool call） 去加载对应技能的完整内容（包括 schema、业务规则、示例 SQL）。
- 加载后再写 SQL，这样就避免了一开始把成千上万张表、规则都塞进上下文导致爆窗、幻觉、成本高。

其具体的运行流程如下：
- 用户提出需求：让系统“为高价值客户写一条 SQL 查询”。
- 智能体先看到可用的技能列表：比如 sales_analytics、inventory_management（相当于可选的知识包/提示词包）。
- 智能体做一次判断：要写这条 SQL，是否需要销售库的表结构？
- 如果需要：就调用 load_skill("sales_analytics") 去加载对应技能。
- 技能加载完成：系统把销售相关的 schema（例如 customers、orders 表）以及一些业务规则/逻辑放进上下文。
- 智能体开始写 SQL：用刚加载的表结构与业务规则来生成查询语句。
输出结果：返回一条符合业务规则、可用的 SQL。

假如使用 Skills 的方式构建多智能体系统，通常分为以下几步：
- 根据任务场景创建对应的 Skills 技能
- 创建 Skills 载入工具（通过 ToolMessage 的方式将 Skills 的具体内容载入上下文中）
- 创建中间件将 Skills 名称和简介动态载入系统提示词中
- 将中间件、工具和记忆组合创建智能体并调用
（进阶）添加内部状态信息确保查询前已加载 Skills

## 2.2 创建 skill

首先我们需要先定义 skill 的格式，这里我们将 skill 拆分成 ：
- name：名称
- description：一句话的简述
- content：实际保存的 Skill 内容

这里的 name 和 description 就是名称和一句话的描述，这两部分就会放到系统提示词里让智能体像选择工具一样进行选用。而 content 就是当智能体决定调用工具来获取到具体内容信息，这部分信息就会在调用后以工具结果的形式展示在上下文中。

In [3]:
from typing import TypedDict

class Skill(TypedDict):
    """A skill that can be progressively disclosed to the agent."""
    name: str
    description: str
    content: str

在设置好了格式后，我们就可以来定义 Skills 了。

这里总共有两个 skill：
- sales_analytics：用于销售数据分析的数据库结构与业务逻辑，包括客户、订单和收入相关内容。
- inventory_management：用于库存管理的数据库结构与业务逻辑，包括商品、仓库和库存数量。

两者的结构都是一样：
- Tables：数据库表结构及字段信息
- Business Logic：相关的业务规则
- Example Query：SQL 查询的示例

In [4]:
SKILLS: list[Skill] = [
    {
        "name": "sales_analytics",
        "description": "Database schema and business logic for sales data analysis including customers, orders, and revenue.",
        "content": """# Sales Analytics Schema

## Tables

### customers
- customer_id (PRIMARY KEY)
- name
- email
- signup_date
- status (active/inactive)
- customer_tier (bronze/silver/gold/platinum)

### orders
- order_id (PRIMARY KEY)
- customer_id (FOREIGN KEY -> customers)
- order_date
- status (pending/completed/cancelled/refunded)
- total_amount
- sales_region (north/south/east/west)

### order_items
- item_id (PRIMARY KEY)
- order_id (FOREIGN KEY -> orders)
- product_id
- quantity
- unit_price
- discount_percent

## Business Logic

**Active customers**: status = 'active' AND signup_date <= CURRENT_DATE - INTERVAL '90 days'

**Revenue calculation**: Only count orders with status = 'completed'. Use total_amount from orders table, which already accounts for discounts.

**Customer lifetime value (CLV)**: Sum of all completed order amounts for a customer.

**High-value orders**: Orders with total_amount > 1000

## Example Query

-- Get top 10 customers by revenue in the last quarter
SELECT
    c.customer_id,
    c.name,
    c.customer_tier,
    SUM(o.total_amount) as total_revenue
FROM customers c
JOIN orders o ON c.customer_id = o.customer_id
WHERE o.status = 'completed'
  AND o.order_date >= CURRENT_DATE - INTERVAL '3 months'
GROUP BY c.customer_id, c.name, c.customer_tier
ORDER BY total_revenue DESC
LIMIT 10;
""",
    },
    {
        "name": "inventory_management",
        "description": "Database schema and business logic for inventory tracking including products, warehouses, and stock levels.",
        "content": """# Inventory Management Schema

## Tables

### products
- product_id (PRIMARY KEY)
- product_name
- sku
- category
- unit_cost
- reorder_point (minimum stock level before reordering)
- discontinued (boolean)

### warehouses
- warehouse_id (PRIMARY KEY)
- warehouse_name
- location
- capacity

### inventory
- inventory_id (PRIMARY KEY)
- product_id (FOREIGN KEY -> products)
- warehouse_id (FOREIGN KEY -> warehouses)
- quantity_on_hand
- last_updated

### stock_movements
- movement_id (PRIMARY KEY)
- product_id (FOREIGN KEY -> products)
- warehouse_id (FOREIGN KEY -> warehouses)
- movement_type (inbound/outbound/transfer/adjustment)
- quantity (positive for inbound, negative for outbound)
- movement_date
- reference_number

## Business Logic

**Available stock**: quantity_on_hand from inventory table where quantity_on_hand > 0

**Products needing reorder**: Products where total quantity_on_hand across all warehouses is less than or equal to the product's reorder_point

**Active products only**: Exclude products where discontinued = true unless specifically analyzing discontinued items

**Stock valuation**: quantity_on_hand * unit_cost for each product

## Example Query

-- Find products below reorder point across all warehouses
SELECT
    p.product_id,
    p.product_name,
    p.reorder_point,
    SUM(i.quantity_on_hand) as total_stock,
    p.unit_cost,
    (p.reorder_point - SUM(i.quantity_on_hand)) as units_to_reorder
FROM products p
JOIN inventory i ON p.product_id = i.product_id
WHERE p.discontinued = false
GROUP BY p.product_id, p.product_name, p.reorder_point, p.unit_cost
HAVING SUM(i.quantity_on_hand) <= p.reorder_point
ORDER BY units_to_reorder DESC;
""",
    },
]

## 2.3 创建 skill 载入工具

在创建完 skill 以后，我们还需要设置一个工具让智能体能够在合适的时候获取 skill 内部对应的 content 内容。

该工具传入的参数是可能的 skill 名称，然后先看看是否存在于 SKILLS 中。假如存在的话就将对应的 content 内容进行返回。

假如没有找到的话，那就把有的 skills 名称都提取出来，然后返回表示并未找到指定的 skill，目前有的 skill 是什么。

In [5]:
from langchain.tools import tool

@tool
def load_skill(skill_name: str) -> str:
    """将某个技能（skill）的完整内容加载到智能体的上下文中。

    当你需要了解如何处理某一类特定请求的详细信息时，请使用该工具。
    该工具会为你提供该技能领域内的完整说明、策略规则以及操作指导。

    参数说明：
        skill_name：要加载的技能名称
        （例如："expense_reporting"、"travel_booking"）
    """
    # 查找并返回请求的技能内容
    for skill in SKILLS:
        if skill["name"] == skill_name:
            return f"已加载技能：{skill_name}\n\n{skill['content']}"

    # 未找到对应技能
    available = ", ".join(s["name"] for s in SKILLS)
    return f"未找到技能 '{skill_name}'。可用技能包括：{available}"

## 2.5 构建 skill 中间件

设置完工具以后，我们还需要设置对应的中间件来动态更新系统提示词的内容。

由于这里也是在调用模型前修改系统提示词相关信息，因此使用的也是 wrap_model_call 中间件：

In [6]:
from langchain.agents.middleware import ModelRequest, ModelResponse, AgentMiddleware
from langchain.messages import SystemMessage
from typing import Callable

class SkillMiddleware(AgentMiddleware):
    """一个用于将技能（skill）描述注入到系统提示词（system prompt）中的中间件。"""
    tools = [load_skill] # 将 load_skill 工具注册为该中间件可用的工具

    def __init__(self):
        """初始化中间件，并基于 SKILLS 生成技能说明提示内容。"""
        skills_list = [] # 从 SKILLS 列表中构建技能说明文本
        for skill in SKILLS:
            skills_list.append(f"- **{skill['name']}**：{skill['description']}")  
        self.skills_prompt = "\n".join(skills_list) # 将所有技能说明拼接成一段文本

    def wrap_model_call(self, request: ModelRequest, handler: Callable[[ModelRequest], ModelResponse]) -> ModelResponse:
        """同步执行：在模型调用前，将技能描述注入到 system prompt 中。"""
        skills_addendum = (f"\n\n## 可用技能（Available Skills）\n\n{self.skills_prompt}\n\n"
            "当你需要处理某一类具体请求的详细信息时，"
            "请使用 load_skill 工具来加载对应技能的完整内容。") # 构建技能说明的补充内容
        new_content = list(request.system_message.content_blocks) + [
            {"type": "text", "text": skills_addendum}] # 将技能说明追加到系统消息的内容块中
        new_system_message = SystemMessage(content=new_content) # 构造新的系统消息
        # 用新的 system message 覆盖原有请求中的 system message
        modified_request = request.override(system_message=new_system_message)
        # 将修改后的请求继续交给下一个处理器（模型）执行
        return handler(modified_request)

## 2.6 创建并调用智能体

当我们将Skills、工具、中间件都打造好以后，我们就可以来将其组合在一起了，这里我们就还需要写入短期记忆内容 InMemorySaver() 和一段简短的系统提示词即可：

In [7]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver
from langchain_community.chat_models import ChatTongyi
import os

model = ChatTongyi(api_key=os.environ.get("DASHSCOPE_API_KEY"), model="qwen-turbo")

# 创建一个支持技能（skill）机制的智能体
agent = create_agent(
    model,
    system_prompt=(
        "你是一个 SQL 查询助手，"
        "负责帮助用户针对企业级业务数据库编写 SQL 查询。"
    ),
    middleware=[SkillMiddleware()],  
    checkpointer=InMemorySaver(),
)

此时我们就可以对该智能体进行调用了，首先我们还是要先创建一个线程 ID：

In [8]:
import uuid

# 为当前这次对话生成一个唯一的线程 ID
thread_id = str(uuid.uuid4())

# 构造本次调用的配置参数
# configurable.thread_id 用于让 agent / checkpointer
# 识别并区分不同的对话线程
config = {"configurable": {"thread_id": thread_id}}

然后我们就可以发出一个查询的请求，了解一下一个月内超过 1000 美元订单的客户有哪些：

In [9]:
# 向智能体提出一个 SQL 查询请求
result = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": (
                    "请编写一条 SQL 查询，用于找出在最近一个月内，"
                    "下过金额超过 1000 美元订单的所有客户"
                ),
            }
        ]
    },
    config
)

# 3. 进阶应用-确保回答问题时已加载对应的 Skills

## 3.1 简介

在前面的版本里，其实存在一个隐患。模型理论上可以：
- 还没加载 sales_analytics
- 就开始写 sales 的 SQL
- 或者乱用 schema

也就是说：
- 我们只是“引导”模型
- 但没有强制约束

而下面我们需要进阶应用的事情是：
- 把“先加载 skill → 再使用相关工具” 
- 从一种提示习惯，升级为系统级规则

## 3.2 创建 State 保存加载信息

为了实现这个功能，首先要做的就是把“已加载了哪些 skills”的信息记录到 agent 的 state 里，然后让后续工具基于 state 决定“能不能用”。

因此需要先定义一个 AgentState 的字段 skills_loaded，用列表的形式将已经载入的 Skills 进行记录：

In [10]:
from langchain.agents.middleware import AgentState
from typing import NotRequired

class CustomState(AgentState):
    skills_loaded: NotRequired[list[str]]

## 3.3 更新 load_skill 工具

有了这个 state 信息后，那我们还需要更新一下 load_skill 工具，使其能够在获取完 skills 里的内容后顺便把 skill_loaded 信息通过 Command 进行更新：

In [14]:
from langchain.tools import ToolRuntime
from langgraph.types import Command
from langchain.messages import ToolMessage

@tool
def load_skill(skill_name: str, runtime: ToolRuntime) -> Command:
    """...省略工具介绍信息..."""
    # 查找并返回对应的技能
    for skill in SKILLS:
        if skill["name"] == skill_name:
            skill_content = f"已加载技能：{skill_name}\n\n{skill['content']}"
            # 更新系统状态，用于记录已加载的技能
            return Command(update={
                    "messages": [ToolMessage(
                            content=skill_content,
                            tool_call_id=runtime.tool_call_id,)],
                    # 在 agent 的 state 中记录已加载的技能名称
                    "skills_loaded": [skill_name]})
    # 未找到对应技能
    available = ", ".join(s["name"] for s in SKILLS)
    return Command(update={"messages": [
                ToolMessage(
                    content=f"未找到技能“{skill_name}”。可用技能包括：{available}",
                    tool_call_id=runtime.tool_call_id,)]})

假如能够找到对应的 skill 的话，那就通过 update 的方法将 skills_load 的信息进行更新。假如没找到的话也还是通过 Command 将未找到的信息进行传回。

## 3.4 添加工具确保只有加载 skill 才能查询
那拥有了 state 以后，我们也可以更进一步去实现只有在“对应 skill 已加载”的前提下，才能查询对应业务领域的内容。

这里我们定义了一个 write_sql_query 的工具来进行实现。

该工具并未真正对 SQL 的格式进行检验，主要是确保在写查询之前检查是否已加载对应的 skill。假如不存在的话就返回错误信息。假如存在就通过校验。

In [15]:
@tool
def write_sql_query(query: str, vertical: str, runtime: ToolRuntime) -> str:
    """...注释部分省略..."""
    # 从运行时状态中读取已经加载过的技能列表
    skills_loaded = runtime.state.get("skills_loaded", [])
    # 如果所需的业务领域 skill 尚未加载，则返回错误提示
    if vertical not in skills_loaded:
        return (
            f"错误：在编写 SQL 查询之前，必须先加载 '{vertical}' 技能，"
            f"以便理解对应的数据库结构。"
            f"请先使用 load_skill('{vertical}') 来加载该 schema。"
        )

    # 校验并格式化 SQL 查询（此处为示例实现）
    return (
        f"{vertical} 业务领域的 SQL 查询如下：\n\n"
        f"```sql\n{query}\n```\n\n"
        f"✓ 查询已通过 {vertical} schema 校验\n"
        f"可以安全地在数据库中执行。"
    )

## 3.5 中间件更新

有了这个 state 和新的 write_sql_query 工具后，我们还需要将其加载到中间件 SkillMiddleware 中。

该中间件主要是把新写入的工具以及对应的内部状态信息进行写入。在这里写入和在创建 agent 时写入都是可以的。

In [16]:
class SkillMiddleware(AgentMiddleware):
    """一个用于将技能（skill）描述注入到系统提示词（system prompt）中的中间件。"""

    state_schema = CustomState  
    tools = [load_skill, write_sql_query]  

    def __init__(self):
        """初始化中间件，并基于 SKILLS 生成技能说明提示内容。"""
        # 从 SKILLS 列表中构建技能说明文本
        skills_list = []
        for skill in SKILLS:
            skills_list.append(
                f"- **{skill['name']}**：{skill['description']}"
            )
        # 将所有技能说明拼接成一段文本
        self.skills_prompt = "\n".join(skills_list)

    def wrap_model_call(
        self,
        request: ModelRequest,
        handler: Callable[[ModelRequest], ModelResponse],
    ) -> ModelResponse:
        """同步执行：在模型调用前，将技能描述注入到 system prompt 中。"""
        # 构建技能说明的补充内容
        skills_addendum = (
            f"\n\n## 可用技能（Available Skills）\n\n{self.skills_prompt}\n\n"
            "当你需要处理某一类具体请求的详细信息时，"
            "请使用 load_skill 工具来加载对应技能的完整内容。"
        )

        # 将技能说明追加到系统消息的内容块中
        new_content = list(request.system_message.content_blocks) + [
            {"type": "text", "text": skills_addendum}
        ]

        # 构造新的系统消息
        new_system_message = SystemMessage(content=new_content)

        # 用新的 system message 覆盖原有请求中的 system message
        modified_request = request.override(system_message=new_system_message)

        # 将修改后的请求继续交给下一个处理器（模型）执行
        return handler(modified_request)

## 3.6 创建智能体并调用

然后我们就能够再次构建 agent 并进行调用：

In [17]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver

# 创建一个支持技能（skill）机制的智能体
agent = create_agent(
    model,
    system_prompt=(
        "你是一个 SQL 查询助手，"
        "负责帮助用户针对企业级业务数据库编写 SQL 查询。"
    ),
    middleware=[SkillMiddleware()],  
    checkpointer=InMemorySaver(),
)

# 向智能体提出一个 SQL 查询请求
result = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": (
                    "请编写一条 SQL 查询，用于找出在最近一个月内，"
                    "下过金额超过 1000 美元订单的所有客户"
                ),
            }
        ]
    },
    config
)

print(result)

{'messages': [HumanMessage(content='请编写一条 SQL 查询，用于找出在最近一个月内，下过金额超过 1000 美元订单的所有客户', additional_kwargs={}, response_metadata={}, id='69494f86-5fef-458a-bbf1-69406ae79689'), AIMessage(content='', additional_kwargs={'tool_calls': [{'function': {'arguments': '{"skill_name": "sales_analytics"}', 'name': 'load_skill'}, 'id': 'call_b587b739fb3044198f22d7', 'index': 0, 'type': 'function'}]}, response_metadata={'model_name': 'qwen-turbo', 'finish_reason': 'tool_calls', 'request_id': '4876a272-db32-4944-8d47-699e87ee41a3', 'token_usage': {'input_tokens': 348, 'output_tokens': 22, 'prompt_tokens_details': {'cached_tokens': 0}, 'total_tokens': 370}}, id='lc_run--019ba24e-f6cf-7a71-afdc-ab23df558979-0', tool_calls=[{'name': 'load_skill', 'args': {'skill_name': 'sales_analytics'}, 'id': 'call_b587b739fb3044198f22d7', 'type': 'tool_call'}], invalid_tool_calls=[]), ToolMessage(content="已加载技能：sales_analytics\n\n# Sales Analytics Schema\n\n## Tables\n\n### customers\n- customer_id (PRIMARY KEY)\n- name

我们可以具体到 LangSmith Studio 中进行进一步的调试。